## SUPERVISED CLASSIFICATION

### FINE TUNING A PRETRAINED BERT MODEL

In [1]:
from datasets import load_dataset

# Prepare data and splits
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
# select the underlying model to use
# then , define the number of labels to predict - necessary for the feedforward neural network that is applied on top of out pretrained model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# load model and tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
# tokenize the data
from transformers import DataCollatorWithPadding

# pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
  """tokenize input data """
  return tokenizer(examples["text"], truncation=True)

# tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
from evaluate import load

# define metrics
def compute_metrics(eval_pred):
  """ calculate f1 score """
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  load_f1 = load("f1")
  f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
  return {"f1":f1}

In [ ]:
# instantiate the trainer
from transformers import TrainingArguments, Trainer

# define training arguments
training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to = "none"
)

In [ ]:
# trainer which executes the training process
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2787764566.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# train and evaluate
trainer.train()
trainer.evaluate()

Step,Training Loss
500,0.416800


{'eval_loss': 0.36028149724006653,
 'eval_model_preparation_time': 0.0048,
 'eval_f1': 0.8434864104967198,
 'eval_runtime': 3.6438,
 'eval_samples_per_second': 292.548,
 'eval_steps_per_second': 18.387,
 'epoch': 1.0}

### FREEZING LAYERS

In [ ]:
# reinitialize the model
# load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# print layer names
for name, param in model.named_parameters():
  print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [ ]:
# freeze everythin except for the classification head
for name, param in model.named_parameters():
  # trainable classification head
  if name.startswith("classifier"):
    param.requires_grad = True
    # freeze everythin else
  else:
    param.requires_grad = False

In [ ]:
# now train the model

from transformers import TrainingArguments, Trainer

# define training arguments
training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to = "none"
)

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# train and evaluate
trainer.train()

/tmp/ipython-input-3772015449.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.699500


TrainOutput(global_step=534, training_loss=0.6990877537245161, metrics={'train_runtime': 33.9741, 'train_samples_per_second': 251.074, 'train_steps_per_second': 15.718, 'total_flos': 227605451772240.0, 'train_loss': 0.6990877537245161, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.6839572787284851,
 'eval_f1': 0.6464995678478824,
 'eval_runtime': 3.8925,
 'eval_samples_per_second': 273.862,
 'eval_steps_per_second': 17.213,
 'epoch': 1.0}

In [ ]:
# lets freeze first 10 encoder blocks of our bert model - everythin else is trainable and will be fine - tuned ----reduces computation
# load model
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# encoder block 11 starts at index 165 and we freeze everythin before that block
for index, (name, param) in enumerate(model.named_parameters()):
  if index < 165:
    param.requires_grad = False

# trainer which executes the training process
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# train and evaluate
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1740128841.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.476100


TrainOutput(global_step=534, training_loss=0.47253813368550845, metrics={'train_runtime': 57.4078, 'train_samples_per_second': 148.586, 'train_steps_per_second': 9.302, 'total_flos': 227605451772240.0, 'train_loss': 0.47253813368550845, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.4100649952888489,
 'eval_f1': 0.8072519083969466,
 'eval_runtime': 3.7699,
 'eval_samples_per_second': 282.766,
 'eval_steps_per_second': 17.772,
 'epoch': 1.0}

## FINE TUNING FOR FEW SHOT CLASSIFICATION

In [ ]:
from setfit import sample_dataset

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# we simulate a few shot setting by sampling 16 examples per class
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=16)

In [ ]:
# we choose a pretrained sentenceTransformer model to fine tune
from setfit import SetFitModel

# load a pretrained sentencet transformer model
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [ ]:
# trainer - by default logistic regression is selected

from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

# define training arguments
args = SetFitTrainingArguments(
    num_iterations=20, # number of text pairs to generate
    num_epochs=3, # number of epochs to use for contrastive learning
)
args.eval_strategy = args.evaluation_strategy

# create trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    args=args,
    metric = "f1"
)

# train and evaluate
trainer.train()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 1280
  Batch size = 16
  Num epochs = 3
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: thesage210 (thesage210-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:263: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:

In [ ]:
trainer.evaluate()

***** Running evaluation *****
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())


{'f1': 0.8454011741682974}

## CONTINUE PRETRAINING WITH MASKED LANGUAGE MODELLING

In [ ]:
# we start by loading bert-base-cased model - and continue pretraining an already pretrained BERT
from transformers import AutoTokenizer, AutoModelForMaskedLM

# load model and tokenizer
model_id = "bert-base-cased"
model = AutoModelForMaskedLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
# we need to tokenize the raw sentences - also remove the labels since this is not a supervised task

def preprocess_function(examples):
  return tokenizer(examples["text"], truncation=True)

# tokenize data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_train = tokenized_train.remove_columns(["label"])
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_test = tokenized_test.remove_columns(["label"])

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
# this time we will have a data collator which will perform the masking of tokens for us.

# here using token masking
from transformers import DataCollatorForLanguageModeling

# define data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15,
    mlm = True
)

In [ ]:
# trainer
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy = "epoch",
    report_to = "none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipython-input-752009111.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# before starting the trainer first save the pretrained tokenizer, then train and then save the model as well
tokenizer.save_pretrained("mlm")

trainer.train()

model.save_pretrained("mlm")

Step,Training Loss
500,2.604500
1000,2.371400
1500,2.306700
2000,2.191000
2500,2.150500
3000,2.089400
3500,2.053600
4000,1.993700
4500,1.987100
5000,1.955500


In [ ]:
from transformers import pipeline

# load and create predictions
mask_filler = pipeline("fill-mask", model="bert-base-cased")
preds = mask_filler("What a horrible [MASK]!")

# print results
for pred in preds:
  print(f">>> {pred['sequence']}")

In [ ]:
# let's see what our updated model predicts
# load and create predictions
mask_filler = pipeline("fill-mask", model="mlm")
preds = mask_filler("What a horrible [MASK]!")

# print results
for pred in preds:
  print(f">>> {pred['sequence']}")

In [ ]:
# next step is to fine tune our updated model on the classification task
# simple do as below and we are good to go
from transformers import AutoModelForSequenceClassification

# fine tune for classification
model = AutoModelForSequenceClassification.from_pretrained("mlm", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("mlm")

## NAMED ENTITY RECOGNITION

In [2]:
# the CoNLL-2003 dataset for NER
dataset = load_dataset("eriktks/conll2003",revision="convert/parquet")

0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [3]:
example = dataset["train"][848]

In [4]:
example

{'id': '848',
 'tokens': ['Dean',
  'Palmer',
  'hit',
  'his',
  '30th',
  'homer',
  'for',
  'the',
  'Rangers',
  '.'],
 'pos_tags': [22, 22, 38, 29, 16, 21, 15, 12, 23, 7],
 'chunk_tags': [11, 12, 21, 11, 12, 12, 13, 11, 12, 0],
 'ner_tags': [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]}

In [5]:
label2id = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-MISC": 7,
    "I-MISC": 8
}

id2label = {index:label for label, index in label2id.items()}
id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC'}

In [6]:
# data is split up into words but not yet tokens
# let's tokenize our data
from transformers import AutoModelForTokenClassification,AutoTokenizer

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    id2label=id2label,
    label2id=label2id
)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# let's explore how the tokenizer would process our exmples

# split individual tokens into sub-tokens
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)

# print results
sub_tokens

['[CLS]',
 'Dean',
 'Palmer',
 'hit',
 'his',
 '30th',
 'home',
 '##r',
 'for',
 'the',
 'Rangers',
 '.',
 '[SEP]']

In [8]:
# we need to align labels - look at the tokens(home, ##r)
# this will tokenize the input and align these tokens with their updated labels during tokenization

def align_labels(examples):
  token_ids = tokenizer(
      examples["tokens"],
      is_split_into_words=True,
      truncation=True
  )

  labels = examples["ner_tags"]
  updated_labels =[]
  for index, label in enumerate(labels):

    # map tokens to their respective words
    word_ids = token_ids.word_ids(batch_index=index)
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
      # the start for a new word
      if word_idx != previous_word_idx:
        previous_word_idx=word_idx
        updated_label = -100 if word_idx is None else label[word_idx]
        label_ids.append(updated_label)

      # special token in -100
      elif word_idx is None:
        label_ids.append(-100)

      # if the lavel is B-XXX we change it to I-XXX
      else:
        updated_label = label[word_idx]
        if updated_label % 2 == 1:
          updated_label += 1
        label_ids.append(updated_label)

    updated_labels.append(label_ids)

  token_ids["labels"] = updated_labels
  return token_ids

tokenized= dataset.map(align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [9]:
# difference between original and updated labels
print(example["tokens"])
print(example["ner_tags"])
print(tokenized["train"][848]["tokens"])
print(tokenized["train"][848]["labels"])

['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.']
[1, 2, 0, 0, 0, 0, 0, 0, 3, 0]
['Dean', 'Palmer', 'hit', 'his', '30th', 'homer', 'for', 'the', 'Rangers', '.']
[-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]


In [10]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [11]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=3f92e5b60e061f337a981f9419a47c19483b7a5fadc0ac6ef443c138886f10b5
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [12]:
# we will use the evaluate package by hugging face to craete a compute_metrics function that allows us to evaluate performance on a token level
# since this time we have multiple predictions per document

import evaluate
import numpy as np

# load sequential evaluation
seq_eval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
  # create predictions
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  true_predictions = []
  true_labels = []

  # document level iterations
  for prediction, label in zip(predictions, labels):

    # token level iteration
    for token_prediction, token_label in zip(prediction, label):
      # we ignore special tokens
      if token_label != -100:
        true_predictions.append(id2label[token_prediction])
        true_labels.append(id2label[token_label])
  results = seq_eval.compute(
      predictions=[true_predictions],
      references=[true_labels]
  )
  return {
      "precision":results["overall_precision"],
      "recall":results["overall_recall"],
      "f1":results["overall_f1"],
      "accuracy":results["overall_accuracy"]
  }

In [13]:
# we need a collator that works with classification on a token level, namely DataCollatorForTokenClassification
from transformers import DataCollatorForTokenClassification

# define data collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [14]:
# training arguments for parameter tuning
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to = "none"
)

# initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

/tmp/ipython-input-1606146873.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.222000


TrainOutput(global_step=878, training_loss=0.16152166831466222, metrics={'train_runtime': 163.367, 'train_samples_per_second': 85.948, 'train_steps_per_second': 5.374, 'total_flos': 351240792638148.0, 'train_loss': 0.16152166831466222, 'epoch': 1.0})

In [16]:
trainer.evaluate()

{'eval_loss': 0.1424957513809204,
 'eval_precision': 0.8425460636515912,
 'eval_recall': 0.8905807365439093,
 'eval_f1': 0.8658977448786367,
 'eval_accuracy': 0.9661524400813097,
 'eval_runtime': 13.4779,
 'eval_samples_per_second': 256.198,
 'eval_steps_per_second': 16.026,
 'epoch': 1.0}

In [15]:
# let's save the model and use it in a pipeline for inference
from transformers import pipeline

# save our fine tuned model
trainer.save_model("ner_model")
# run the inference on the fine-tuned model
token_classifier = pipeline(
    "token-classification",
    model="ner_model"
)
token_classifier("My name is Sanyam")

Device set to use cuda:0


[{'entity': 'B-PER',
  'score': np.float32(0.99160403),
  'index': 4,
  'word': 'San',
  'start': 11,
  'end': 14},
 {'entity': 'I-PER',
  'score': np.float32(0.9797769),
  'index': 5,
  'word': '##yam',
  'start': 14,
  'end': 17}]